# 03 - Heston Monte Carlo Engine

**Objective:** Simulate Heston spot/variance paths and validate the engine against the Black-Scholes limiting case.

**Inputs:** The cleaned AAPL surface when available; otherwise fallback demonstration parameters.

**Outputs:** Sample path figures and Black-Scholes-limit validation metrics.

**Key takeaway:** Setting volatility-of-volatility to zero should collapse Heston to Black-Scholes, giving a numerical implementation check.



In [ ]:
from pathlib import Path
import os
import sys
import tempfile

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "heston_matplotlib_cache"))

FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.black_scholes import price
from src.heston import HestonParams, get_torch_device, price_european_mc, simulate_heston_paths, pathwise_delta

In [ ]:
TICKER = "AAPL"
checked_path = PROJECT_ROOT / "data" / "processed" / f"{TICKER}_surface_checked.csv"
sample_checked_path = PROJECT_ROOT / "data" / "sample" / f"{TICKER}_canonical_surface_checked.csv"

if checked_path.exists():
    df = pd.read_csv(checked_path, parse_dates=["expiry"])
    spot = float((df["strike"] / df["moneyness"]).median())
    rate = float(df["r"].dropna().median()) if "r" in df else 0.03
    dividend = float(df["q"].dropna().median()) if "q" in df else 0.0
    print(f"Loaded {checked_path.relative_to(PROJECT_ROOT)}")
elif sample_checked_path.exists():
    df = pd.read_csv(sample_checked_path, parse_dates=["expiry"])
    spot = float((df["strike"] / df["moneyness"]).median())
    rate = float(df["r"].dropna().median()) if "r" in df else 0.03
    dividend = float(df["q"].dropna().median()) if "q" in df else 0.0
    print(f"Loaded {sample_checked_path.relative_to(PROJECT_ROOT)}")
else:
    spot, rate, dividend = 100.0, 0.03, 0.0

device = get_torch_device(prefer_gpu=True)
print(f"Using device: {device}")
print(f"spot={spot:.4f}, r={rate:.4f}, q={dividend:.4f}")



## Simulate Heston Paths

The parameters below are a transparent starting point. Calibration in notebook 04 estimates them from the observed market smile.

In [ ]:
params = HestonParams(v0=0.04, kappa=2.0, theta=0.04, sigma=0.30, rho=-0.70)
maturity = 1.0
n_steps = 252
n_paths = 50_000

spot_paths, variance_paths = simulate_heston_paths(
    spot=spot,
    rate=rate,
    dividend=dividend,
    params=params,
    maturity=maturity,
    n_steps=n_steps,
    n_paths=n_paths,
    seed=42,
    device=device,
)
print(spot_paths.shape, variance_paths.shape)

In [ ]:
time_grid = np.linspace(0, maturity, n_steps + 1)
spot_sample = spot_paths[:10].detach().cpu().numpy()
var_sample = variance_paths[:10].detach().cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
axes[0].plot(time_grid, spot_sample.T, alpha=0.7)
axes[0].set_title("Heston spot paths")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Spot")

axes[1].plot(time_grid, var_sample.T, alpha=0.7)
axes[1].axhline(params.theta, linestyle="--", color="red", label="theta")
axes[1].set_title("Heston variance paths")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Variance")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "heston_sample_paths.png", dpi=240)
plt.close()

## Price and Validate

When the volatility of volatility is zero and $v_0=\theta$, the Heston model collapses to Black-Scholes. That gives a direct implementation check.

In [ ]:
strike = spot
heston_price = price_european_mc(spot_paths, strike=strike, maturity=maturity, rate=rate, side="C").item()
heston_delta = pathwise_delta(spot_paths, spot=spot, strike=strike, maturity=maturity, rate=rate, side="C").item()
print(f"Heston ATM call price: {heston_price:.4f}")
print(f"Pathwise ATM call delta: {heston_delta:.4f}")

In [ ]:
bs_vol = 0.20
bs_params = HestonParams(v0=bs_vol**2, kappa=2.0, theta=bs_vol**2, sigma=0.0, rho=0.0)
bs_paths, _ = simulate_heston_paths(
    spot=spot,
    rate=rate,
    dividend=dividend,
    params=bs_params,
    maturity=maturity,
    n_steps=128,
    n_paths=80_000,
    seed=7,
    device=device,
)
mc_bs_price = price_european_mc(bs_paths, strike=strike, maturity=maturity, rate=rate, side="C").item()
analytic_bs_price = price(spot, strike, maturity, rate, bs_vol, "C", dividend)
print(f"MC Black-Scholes-limit price: {mc_bs_price:.4f}")
print(f"Analytical Black-Scholes price: {analytic_bs_price:.4f}")
print(f"Difference: {mc_bs_price - analytic_bs_price:.4f}")

## Takeaway

The Monte Carlo engine passes a Black-Scholes limiting-case check, so notebook 04 can use it as the pricing engine for calibration.

